# Miscellaneous

Patterns and APIs that don't fit the per-field schema customization story in `02_example_customize`. Each section is self-contained — read whichever one you need.

## Change Global Type Defaults

By default, `annotate_xeval` assigns `exact` to every string field and `numeric` to every number field. If you want a different default for an entire type — applied to every matching field without touching them one by one — use `set_type_default` before calling `annotate_xeval`.

Per-field `x-eval-compare` still wins over the type default, so this is a "set the baseline, override where needed" pattern. Useful when you have many fields of one type and the built-in default isn't right for your domain.

**Caveat:** `set_type_default` mutates process-wide state. Always pair it with `reset_type_defaults()` so it doesn't leak into other evaluations in the same session.

**Example:** make every number field default to a comparator that rounds to int before comparing (useful when your domain treats numeric values as whole units).

In [ ]:
# Minimal self-contained setup
GOLD = [
    {"material": "Silicon", "temperature": 300.0},
    {"material": "Gold",    "temperature": 200.0},
]
EXTRACTED = [
    {"material": "Silicon", "temperature": 300.4},
    {"material": "Gold",    "temperature": 200.0},
]

from struct_extract_eval import (
    infer_schema,
    annotate_xeval,
    evaluate,
    set_type_default,
    reset_type_defaults,
)
from struct_extract_eval.core.comparators.comparator import ComparatorResult
from struct_extract_eval.core.comparators.registry import register, _clear_registry


def compare_numeric_int(gold, extracted, params):
    """Round both values to the nearest int, then compare."""
    try:
        g, e = round(float(gold)), round(float(extracted))
        return ComparatorResult(score=1.0 if g == e else 0.0, comparator="numeric_int")
    except (ValueError, TypeError):
        return ComparatorResult(score=0.0, comparator="numeric_int")


_clear_registry()
register("numeric_int", compare_numeric_int)

# Make every number field default to numeric_int
set_type_default("number", "numeric_int")

schema = infer_schema(GOLD)
annotate_xeval(schema)

print("Number fields now default to 'numeric_int':")
for name, prop in schema["properties"].items():
    print(f"  {name}: {prop.get('x-eval-compare')}")

# Evaluate -- temperature 300.4 vs 300.0 both round to 300, so it matches
run = evaluate(GOLD, EXTRACTED, schema=schema)
print(f"\ntemperature: score={run.per_field['temperature'].mean_score:.2f}")
print("  300.4 vs 300.0 -> round to 300 vs 300 -> match")

# Always reset so this doesn't leak into later evaluations
reset_type_defaults()
print("\n(Defaults reset.)")